In [1]:

import pandas as pd
import spacy
from spellchecker import SpellChecker
import re
import importlib
import utils
import calculate_linguisticFeatures

importlib.reload(utils)
importlib.reload(calculate_linguisticFeatures)

<module 'calculate_linguisticFeatures' from 'c:\\Projetos\\mineracao_arquivos_epstein_2026\\calculate_linguisticFeatures.py'>

In [2]:

def count_misspellings(doc):
    """Identifica palavras com erros ortográficos em um texto"""
    
    spell = SpellChecker(language='en')  # Definir o idioma como portuguêsn
    
    # Conta o número de erros ortográficos para cada texto na lista
    misspellings_counts = len(spell.unknown([token.text for token in doc if token.is_alpha]))
    
    return misspellings_counts

def check_and_load_spacy_model(model_name):
    """Verifica se o modelo está instalado, caso contrário, instala automaticamente.
    """
    
    # Verifica se o modelo está disponível
    if model_name not in spacy.util.get_installed_models():
        print(f"Modelo {model_name} não encontrado. Instalando...")
        spacy.cli.download(model_name)
    
    # Carrega o modelo
    nlp = spacy.load(model_name)
    
    return nlp

In [7]:
dt_raw = utils.Utils.load('/datasets',12)

In [ ]:
# dt1 = pd.read_csv('datasets1.csv', sep='|', index_col=0)
# dt2 = pd.read_csv('datasets2.csv', sep='|', index_col=0)
# pd.concat([dt1, dt2])[['dataset', 'file', 'content', 'file_type', 'len', 'count_tokens',
#        'count_misspellings', 'count_sentences']].to_csv('datasets-misspellings.csv', sep='|')

In [4]:
dt_raw['file_type'] = dt_raw['file'].apply(lambda x: x.split('.')[-1])
dt_raw['file_type'] = dt_raw['file_type'].replace({
    'mp4':'video',
    'avi':'video',
    'mp3':'audio',
    'm4a':'audio',
    'xlsx': 'sheet',
    'xls': 'sheet',
    'csv': 'sheet',
})

#PDFs sem texto são todos imagens
dt_raw.loc[dt_raw['content'].isna(), 'file_type'] = 'image'
dt_raw['file_type'].value_counts().to_frame().style

,count
file_type,
pdf,12222
image,2363
video,420
sheet,16
audio,4


In [5]:
dt = dt_raw.loc[dt_raw['file_type'] == 'pdf']

In [6]:
linguistics = calculate_linguisticFeatures.linguisticFeatures()

In [7]:
# linguistics.nlpModel.max_length = 1257000

In [10]:
dt['len'] = dt['content'].apply(lambda x: len(x))
dt = dt.sort_values('len', ascending=False)
dt = dt.loc[dt['len'] < 100000]

In [11]:
dt[['count_tokens', 'count_misspellings', 'count_sentences']] = linguistics.get(
    dt['content'],
    'count_tokens',
    'count_misspellings',
    'count_sentences',
)

MemoryError: Unable to allocate 2.34 GiB for an array with shape (2177201, 288) and data type float32

In [ ]:
print('arquivos sem texto: ', dt['content'].isna().sum())
print((dt['content'].size/dt['content'].isna().sum()).round(2), '%')

arquivos sem texto:  2363
6.37 %
